# Setlist AI Composer

Workflow
1. Google Doc (canonical storage)
2. Parser (doc → structured list)
3. LLM (command → action JSON)
4. Executor (mutate list)
5. Serializer (list → doc string)
6. Overwrite Google Doc

In [ ]:
#Install libraries
import os
import json
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build


In [10]:
# Authenticate to Google

# Full access for setlist agent (read + write)
SCOPES = [
    "https://www.googleapis.com/auth/documents",
    "https://www.googleapis.com/auth/drive"
]

def authenticate_google():
    creds = None

    # Load saved token if it exists
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    # If no valid credentials, run OAuth flow
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json",
                SCOPES
            )
            creds = flow.run_local_server(port=0)

        # Save credentials for future runs
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    # Build API clients (this is what your agent will use later)
    docs_service = build("docs", "v1", credentials=creds)
    drive_service = build("drive", "v3", credentials=creds)

    return docs_service, drive_service


# Run authentication
docs_service, drive_service = authenticate_google()

print("Google authentication successful")


# Read Google Doc
def read_google_doc(docs_service, document_id):
    doc = docs_service.documents().get(documentId=document_id).execute()

    content = doc.get("body").get("content")

    text = ""

    for block in content:
        if "paragraph" in block:
            elements = block["paragraph"]["elements"]
            for el in elements:
                if "textRun" in el:
                    text += el["textRun"]["content"]

    return text


# Usage
DOCUMENT_ID = "10v3rXz9N8SJjl8_p_X6uuDSnYsrXQY8j2wHm8uQDz4E"

doc_text = read_google_doc(docs_service, DOCUMENT_ID)

print(doc_text)


# Parse the input data
def parse_setlist(text):
    lines = text.strip().split("\n")

    songs = []

    for line in lines:
        # skip headers or empty lines
        if "Setlist" in line or not line.strip():
            continue

        parts = [p.strip() for p in line.split("|")]

        if len(parts) == 3:
            song, artist, key = parts

            # clean key prefix if needed ("Key: G" → "G")
            if "Key:" in key:
                key = key.replace("Key:", "").strip()

            songs.append({
                "song": song,
                "artist": artist,
                "key": key
            })

    return songs


# test it
structured_setlist = parse_setlist(doc_text)

print(structured_setlist)



# Serialize setlist back into Google Doc format
def serialize_setlist(songs):
    lines = ["Setlist:\n"]

    for song in songs:
        line = f"{song['song']} | {song['artist']} | Key: {song['key']}"
        lines.append(line)

    return "\n".join(lines)


# Overwrite Google Doc
def overwrite_google_doc(docs_service, document_id, text):
    doc = docs_service.documents().get(documentId=document_id).execute()

    content = doc.get("body").get("content")
    end_index = content[-1]["endIndex"] - 1  # safe last position

    requests = [
        {
            "deleteContentRange": {
                "range": {
                    "startIndex": 1,
                    "endIndex": end_index
                }
            }
        },
        {
            "insertText": {
                "location": {
                    "index": 1
                },
                "text": text
            }
        }
    ]

    docs_service.documents().batchUpdate(
        documentId=document_id,
        body={"requests": requests}
    ).execute()


# 1. serialize structured state
new_doc_text = serialize_setlist(structured_setlist)

# 2. write back to Google Doc
overwrite_google_doc(docs_service, DOCUMENT_ID, new_doc_text)

print("Doc updated successfully")

Google authentication successful
Setlist:

Let’s Stay together | Al Green | Key: G
Sugar | Stanley Turrentine | Key: Cm
Ascension | Maxwell | Key: D

[{'song': 'Let’s Stay together', 'artist': 'Al Green', 'key': 'G'}, {'song': 'Sugar', 'artist': 'Stanley Turrentine', 'key': 'Cm'}, {'song': 'Ascension', 'artist': 'Maxwell', 'key': 'D'}]
Doc updated successfully
